<a href="https://colab.research.google.com/github/Mahalakshmierugu/email-spam-detection/blob/main/QML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install pennylane pennylane-lightning scikit-learn opencv-python


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.6/935.6 kB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 102.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 12.3 MB/s eta 0:00:00


In [ ]:
import os
import cv2
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report


In [ ]:
def load_images_cnn(data_dir, img_size=64):
    X = []
    y = []

    for label, category in enumerate(["NORMAL", "PNEUMONIA"]):
        folder = os.path.join(data_dir, "train", category)
        for img_name in os.listdir(folder):
            img_path = os.path.join(folder, img_name)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

            if img is not None:
                img = cv2.resize(img, (img_size, img_size))
                img = img / 255.0  # normalize
                X.append(img)
                y.append(label)

    X = np.array(X).reshape(-1, img_size, img_size, 1)
    y = np.array(y)
    return X, y


In [ ]:
# Load images again
DATA_DIR = '/content/drive/MyDrive/chest x-ray'
X, y = load_images_cnn(DATA_DIR)

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape, y_train.shape)


(4172, 64, 64, 1) (4172,)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Reshape X_train and X_test to 2D for StandardScaler
X_train_reshaped = X_train.reshape(X_train.shape[0], -1)
X_test_reshaped = X_test.reshape(X_test.shape[0], -1)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_reshaped)
X_test = scaler.transform(X_test_reshaped)


In [ ]:
X_train = X_train[:60]
y_train = y_train[:60]
X_test = X_test[:20]
y_test = y_test[:20]


In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=4)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)


In [ ]:
from tensorflow.keras import models, layers

IMG_SIZE = 64 # Define IMG_SIZE

cnn = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 1)),
    layers.MaxPooling2D(),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(1, activation='sigmoid') # Changed to 1 neuron with sigmoid activation for binary classification
])

cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
import pennylane as qml

n_qubits = 4
dev = qml.device("default.qubit", wires=n_qubits)

def feature_map(x):
    for i in range(n_qubits):
        qml.RY(x[i], wires=i)

@qml.qnode(dev)
def qkernel(x1, x2):
    feature_map(x1)
    qml.adjoint(feature_map)(x2)
    return qml.probs(wires=range(n_qubits))

def quantum_kernel(x1, x2):
    return qkernel(x1, x2)[0]


/usr/local/lib/python3.12/dist-packages/pennylane/__init__.py:209: RuntimeWarning: PennyLane is not yet compatible with JAX versions > 0.6.2. You have version 0.7.2 installed. Please downgrade JAX to 0.6.2 to avoid runtime errors using python -m pip install jax~=0.6.0 jaxlib~=0.6.0
  warnings.warn(


In [ ]:
def kernel_matrix(X1, X2):
    return np.array([[quantum_kernel(x1, x2) for x2 in X2] for x1 in X1])

K_train_qpca = kernel_matrix(X_train_pca, X_train_pca)
K_test_qpca = kernel_matrix(X_test_pca, X_train_pca)


In [ ]:
from sklearn.svm import SVC

qsvm = SVC(kernel='precomputed')
qsvm.fit(K_train_qpca, y_train)


SVC(kernel='precomputed')

In [ ]:
y_pred = qsvm.predict(K_test_qpca)

print("QSVM Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


QSVM Accuracy: 0.75
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         5
           1       0.75      1.00      0.86        15

    accuracy                           0.75        20
   macro avg       0.38      0.50      0.43        20
weighted avg       0.56      0.75      0.64        20



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
cnn = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 1)),
    layers.MaxPooling2D(),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(1, activation='sigmoid')  # ✅ IMPORTANT
])

cnn.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
IMG_SIZE = 64

def load_images_cnn(data_dir):
    X, y = [], []
    labels = {"NORMAL": 0, "PNEUMONIA": 1}

    for label in labels:
        path = os.path.join(data_dir, "train", label) # Added "train" subdirectory
        for img in os.listdir(path):
            try:
                img_path = os.path.join(path, img)
                image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                image = cv2.resize(image, (IMG_SIZE, IMG_SIZE))
                image = image / 255.0
                X.append(image)
                y.append(labels[label])
            except:
                pass

    X = np.array(X)[..., np.newaxis]  # ✅ (N, 64, 64, 1)
    y = np.array(y)
    return X, y


In [ ]:
X, y = load_images_cnn(DATA_DIR)

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape)  # MUST be (N, 64, 64, 1)
cnn = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 1)),
    layers.MaxPooling2D(),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])


(4172, 64, 64, 1)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
cnn.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
cnn.fit(
    X_train, y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.1
)

Epoch 1/5
118/118 ━━━━━━━━━━━━━━━━━━━━ 25s 187ms/step - accuracy: 0.7885 - loss: 0.4699 - val_accuracy: 0.9402 - val_loss: 0.1707
Epoch 2/5
118/118 ━━━━━━━━━━━━━━━━━━━━ 41s 187ms/step - accuracy: 0.9424 - loss: 0.1392 - val_accuracy: 0.9569 - val_loss: 0.1194
Epoch 3/5
118/118 ━━━━━━━━━━━━━━━━━━━━ 24s 202ms/step - accuracy: 0.9578 - loss: 0.1107 - val_accuracy: 0.9474 - val_loss: 0.1186
Epoch 4/5
118/118 ━━━━━━━━━━━━━━━━━━━━ 41s 202ms/step - accuracy: 0.9720 - loss: 0.0760 - val_accuracy: 0.9713 - val_loss: 0.0709
Epoch 5/5
118/118 ━━━━━━━━━━━━━━━━━━━━ 24s 201ms/step - accuracy: 0.9706 - loss: 0.0734 - val_accuracy: 0.9641 - val_loss: 0.0783


In [ ]:
import tensorflow as tf

feature_extractor = tf.keras.Model(
    inputs=cnn.inputs[0],
    outputs=cnn.layers[-2].output
)

X_train_feat = feature_extractor.predict(X_train)
X_test_feat = feature_extractor.predict(X_test)

print(X_train_feat.shape)  # (N, 128)

131/131 ━━━━━━━━━━━━━━━━━━━━ 7s 52ms/step
33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 42ms/step
(4172, 128)


In [ ]:
X_train_feat = X_train_feat[:50]
y_train = y_train[:50]

X_test_feat = X_test_feat[:20]
y_test = y_test[:20]


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_feat = scaler.fit_transform(X_train_feat)
X_test_feat = scaler.transform(X_test_feat)


In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=4)   # 4 features = 4 qubits
X_train_pca = pca.fit_transform(X_train_feat)
X_test_pca = pca.transform(X_test_feat)

print(X_train_pca.shape)   # (50, 4)


(50, 4)


In [ ]:
import pennylane as qml
import numpy as np

n_qubits = 4
dev = qml.device("default.qubit", wires=n_qubits)


In [ ]:
def feature_map(x):
    for i in range(n_qubits):
        qml.RY(x[i], wires=i)


In [ ]:
@qml.qnode(dev)
def quantum_kernel_circuit(x1, x2):
    feature_map(x1)
    qml.adjoint(feature_map)(x2)
    return qml.probs(wires=range(n_qubits))


In [ ]:
def quantum_kernel(x1, x2):
    return quantum_kernel_circuit(x1, x2)[0]


In [ ]:
def kernel_matrix(X1, X2):
    return np.array([[quantum_kernel(x1, x2) for x2 in X2] for x1 in X1])

K_train = kernel_matrix(X_train_pca, X_train_pca)
K_test = kernel_matrix(X_test_pca, X_train_pca)


In [ ]:
from sklearn.svm import SVC

qsvm = SVC(kernel='precomputed')
qsvm.fit(K_train, y_train)


SVC(kernel='precomputed')

In [ ]:
y_pred = qsvm.predict(K_test)

from sklearn.metrics import accuracy_score, classification_report

print("Hybrid CNN + QPCA + QSVM Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Hybrid CNN + QPCA + QSVM Accuracy: 0.8
              precision    recall  f1-score   support

           0       1.00      0.20      0.33         5
           1       0.79      1.00      0.88        15

    accuracy                           0.80        20
   macro avg       0.89      0.60      0.61        20
weighted avg       0.84      0.80      0.75        20



In [ ]:
import numpy as np
print("Train class distribution:", np.bincount(y_train))


Train class distribution: [11 39]


In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(enumerate(class_weights))
print(class_weights)


{0: np.float64(2.272727272727273), 1: np.float64(0.6410256410256411)}


In [ ]:
X_train = X_train[:50]
cnn.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.1,
    class_weight=class_weights
)

Epoch 1/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 202ms/step - accuracy: 1.0000 - loss: 0.0283 - val_accuracy: 1.0000 - val_loss: 1.6011e-04
Epoch 2/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 145ms/step - accuracy: 1.0000 - loss: 0.0252 - val_accuracy: 1.0000 - val_loss: 1.7446e-04
Epoch 3/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 142ms/step - accuracy: 1.0000 - loss: 0.0209 - val_accuracy: 1.0000 - val_loss: 1.9616e-04
Epoch 4/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 161ms/step - accuracy: 1.0000 - loss: 0.0171 - val_accuracy: 1.0000 - val_loss: 1.5632e-04
Epoch 5/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 140ms/step - accuracy: 1.0000 - loss: 0.0128 - val_accuracy: 1.0000 - val_loss: 9.2910e-05
Epoch 6/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 154ms/step - accuracy: 1.0000 - loss: 0.0086 - val_accuracy: 1.0000 - val_loss: 4.2288e-05
Epoch 7/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 144ms/step - accuracy: 1.0000 - loss: 0.0060 - val_accuracy: 1.0000 - val_loss: 2.5468e-05
Epoch 8/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 142ms/step - accuracy: 1.0000 - loss: 0.0058 - val_

In [ ]:
print("Hybrid CNN + QPCA + QSVM Accuracy:", accuracy_score(y_test, y_pred))


Hybrid CNN + QPCA + QSVM Accuracy: 0.8
